# Notebook : 02_zone_remapping.ipynb
# TaaSim — Semaine 1 : Remapping Porto → Casablanca

CELLULE 1 — Bounding boxes des deux villes

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import folium
import json

spark = SparkSession.builder \
    .appName("TaaSim-Remapping") \
    .config("spark.driver.memory", "4g") \
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "taasim") \
    .config("spark.hadoop.fs.s3a.secret.key", "taasim123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print("✅ Spark prêt avec MinIO")
# ===============================
# Bounding boxes des villes
# ===============================

# Bounding box de Porto (Portugal)
# Source : OpenStreetMap / Geofabrik Portugal
PORTO_LON_MIN = -8.7327
PORTO_LON_MAX = -8.5539
PORTO_LAT_MIN = 41.0527
PORTO_LAT_MAX = 41.2370

# Bounding box de Casablanca (Maroc)
# Source : OpenStreetMap / Geofabrik Morocco
CASA_LON_MIN = -7.7000
CASA_LON_MAX = -7.4800
CASA_LAT_MIN = 33.4800
CASA_LAT_MAX = 33.6800

# ===============================
# Affichage des bounding boxes
# ===============================
print("Porto → lon:", PORTO_LON_MIN, "→", PORTO_LON_MAX, "| lat:", PORTO_LAT_MIN, "→", PORTO_LAT_MAX)
print("Casa → lon:", CASA_LON_MIN, "→", CASA_LON_MAX, "| lat:", CASA_LAT_MIN, "→", CASA_LAT_MAX)

✅ Spark prêt avec MinIO
Porto → lon: -8.7327 → -8.5539 | lat: 41.0527 → 41.237
Casa → lon: -7.7 → -7.48 | lat: 33.48 → 33.68


In [7]:
!pip install -r /home/jovyan/work/notebooks/requirements.txt

CELLULE 2 — Fonctions de transformation

In [3]:
# Formule : coord_casa = coord_casa_min + (coord_porto - coord_porto_min)
#           / (coord_porto_max - coord_porto_min) * (coord_casa_max - coord_casa_min)
 
def transform_lon(lon_porto):
    """Transforme une longitude Porto → longitude Casablanca"""
    ratio = (lon_porto - PORTO_LON_MIN) / (PORTO_LON_MAX - PORTO_LON_MIN)
    return CASA_LON_MIN + ratio * (CASA_LON_MAX - CASA_LON_MIN)
 
def transform_lat(lat_porto):
    """Transforme une latitude Porto → latitude Casablanca"""
    ratio = (lat_porto - PORTO_LAT_MIN) / (PORTO_LAT_MAX - PORTO_LAT_MIN)
    return CASA_LAT_MIN + ratio * (CASA_LAT_MAX - CASA_LAT_MIN)
 
# Test rapide
porto_center = (-8.6110, 41.1496)   # centre de Porto
casa_expected = (-7.5830, 33.5680)  # centre de Casablanca approximatif
 
lon_transformed = transform_lon(porto_center[0])
lat_transformed = transform_lat(porto_center[1])
print(f"Centre Porto : {porto_center}")
print(f"Transformé   : ({lon_transformed:.4f}, {lat_transformed:.4f})")
print(f"Attendu      : {casa_expected}")

Centre Porto : (-8.611, 41.1496)
Transformé   : (-7.5503, 33.5852)
Attendu      : (-7.583, 33.568)


CELLULE 3 — UDF PySpark pour le POLYLINE

In [5]:
# Le POLYLINE est un JSON array [[lon1,lat1], [lon2,lat2], ...]
# On doit transformer chaque point
 
def remap_polyline(polyline_str):
    """Transforme tous les points GPS d'un trajet Porto → Casablanca"""
    if polyline_str is None or polyline_str == "[]":
        return None
    try:
        points = json.loads(polyline_str)
        remapped = []
        for pt in points:
            lon_porto, lat_porto = pt[0], pt[1]
            lon_casa = CASA_LON_MIN + (lon_porto - PORTO_LON_MIN) / \
                       (PORTO_LON_MAX - PORTO_LON_MIN) * (CASA_LON_MAX - CASA_LON_MIN)
            lat_casa = CASA_LAT_MIN + (lat_porto - PORTO_LAT_MIN) / \
                       (PORTO_LAT_MAX - PORTO_LAT_MIN) * (CASA_LAT_MAX - CASA_LAT_MIN)
            remapped.append([round(lon_casa, 6), round(lat_casa, 6)])
        return json.dumps(remapped)
    except Exception:
        return None
 
# Enregistrer comme UDF Spark
remap_udf = F.udf(remap_polyline, StringType())

CELLULE 4 — Table de mapping des zones

In [6]:
# Porto a 22 zones numérotées dans ORIGIN_CALL
# On les mappe aux 16 arrondissements de Casablanca
 
zone_mapping_data = [
    # (zone_porto, arrondissement_casa, zone_type, population_density, centroid_lon, centroid_lat)
    (1,  1,  "Ain Sebaa",        "industrial",      -7.5560, 33.6010),
    (2,  2,  "Al Fida",          "residential",     -7.6200, 33.5900),
    (3,  3,  "Anfa",             "commercial",      -7.6500, 33.5800),
    (4,  4,  "Ben M'Sick",       "residential",     -7.5900, 33.5600),
    (5,  5,  "Bernoussi",        "residential",     -7.5450, 33.6100),
    (6,  6,  "Bouskoura",        "residential",     -7.6500, 33.4900),
    (7,  7,  "El Fida-Mers Sultan", "commercial",   -7.6000, 33.5850),
    (8,  8,  "Hay Hassani",      "residential",     -7.6700, 33.5450),
    (9,  9,  "Hay Mohammadi",    "industrial",      -7.5700, 33.5750),
    (10, 10, "Maarif",           "commercial",      -7.6400, 33.5700),
    (11, 11, "Moulay Rachid",    "residential",     -7.5850, 33.5400),
    (12, 12, "Roches Noires",    "transit_hub",     -7.5800, 33.6050),
    (13, 13, "Sidi Belyout",     "commercial",      -7.6100, 33.5950),
    (14, 14, "Sidi Moumen",      "residential",     -7.5300, 33.5900),
    (15, 15, "Sidi Othmane",     "residential",     -7.5650, 33.5500),
    (16, 16, "Ain Diab",         "commercial",      -7.6800, 33.5950),
    # Les zones Porto 17-22 se redistribuent sur les zones Casa 1-6
    (17, 1,  "Ain Sebaa",        "industrial",      -7.5560, 33.6010),
    (18, 5,  "Bernoussi",        "residential",     -7.5450, 33.6100),
    (19, 13, "Sidi Belyout",     "commercial",      -7.6100, 33.5950),
    (20, 3,  "Anfa",             "commercial",      -7.6500, 33.5800),
    (21, 10, "Maarif",           "commercial",      -7.6400, 33.5700),
    (22, 16, "Ain Diab",         "commercial",      -7.6800, 33.5950),
]
 
schema_mapping = StructType([
    StructField("zone_porto",    IntegerType()),
    StructField("zone_id",       IntegerType()),
    StructField("zone_name",     StringType()),
    StructField("zone_type",     StringType()),
    StructField("centroid_lon",  DoubleType()),
    StructField("centroid_lat",  DoubleType()),
])
 
df_zones = spark.createDataFrame(zone_mapping_data, schema=schema_mapping)
df_zones.show(truncate=False)
 
# Sauvegarder en CSV local pour le starter kit
df_zones.toPandas().to_csv("/home/jovyan/work/data/zone_mapping.csv", index=False)
print("zone_mapping.csv sauvegardé")

+----------+-------+-------------------+-----------+------------+------------+
|zone_porto|zone_id|zone_name          |zone_type  |centroid_lon|centroid_lat|
+----------+-------+-------------------+-----------+------------+------------+
|1         |1      |Ain Sebaa          |industrial |-7.556      |33.601      |
|2         |2      |Al Fida            |residential|-7.62       |33.59       |
|3         |3      |Anfa               |commercial |-7.65       |33.58       |
|4         |4      |Ben M'Sick         |residential|-7.59       |33.56       |
|5         |5      |Bernoussi          |residential|-7.545      |33.61       |
|6         |6      |Bouskoura          |residential|-7.65       |33.49       |
|7         |7      |El Fida-Mers Sultan|commercial |-7.6        |33.585      |
|8         |8      |Hay Hassani        |residential|-7.67       |33.545      |
|9         |9      |Hay Mohammadi      |industrial |-7.57       |33.575      |
|10        |10     |Maarif             |commercial |

CELLULE 5 — Appliquer le remapping sur un échantillon

In [7]:
df = spark.read.csv("s3a://raw/porto-trips/train.csv", header=True, inferSchema=True)
 
# Filtrer les trajets valides
df_clean = df.filter(F.col("MISSING_DATA") == "False") \
             .filter(F.col("POLYLINE") != "[]") \
             .filter(F.col("POLYLINE").isNotNull())
 
# Appliquer la transformation sur un sample de 10 000 lignes pour tester
df_sample = df_clean.sample(0.005)   # ~8 500 lignes pour tester
 
df_remapped = df_sample.withColumn("POLYLINE_CASA", remap_udf(F.col("POLYLINE")))
 
# Extraire le premier point de chaque trajet pour visualiser
def get_first_point(polyline_str):
    """Extrait le premier point GPS d'un POLYLINE"""
    if not polyline_str:
        return None
    try:
        pts = json.loads(polyline_str)
        if pts:
            return [pts[0][1], pts[0][0]]  # [lat, lon] pour folium
    except:
        return None
    return None
 
get_first_udf = F.udf(get_first_point, ArrayType(DoubleType()))
 
df_viz = df_remapped \
    .withColumn("start_casa", get_first_udf(F.col("POLYLINE_CASA"))) \
    .filter(F.col("start_casa").isNotNull()) \
    .select("TAXI_ID", "CALL_TYPE", "start_casa") \
    .limit(500)
 
points_pdf = df_viz.toPandas()
print(f"Points à visualiser : {len(points_pdf)}")
print(points_pdf.head(3))

Points à visualiser : 500
    TAXI_ID CALL_TYPE              start_casa
0  20000671         C   [33.585987, -7.55769]
1  20000665         C  [33.585977, -7.551555]
2  20000167         A  [33.584063, -7.552641]


CELLULE 6 — Carte Folium sur OpenStreetMap

In [13]:
colors_call = {"A": "blue", "B": "green", "C": "red"}
 
# Carte centrée sur Casablanca
m = folium.Map(
    location=[33.5731, -7.5898],
    zoom_start=12,
    tiles="OpenStreetMap"
)
 
# Ajouter les zones (arrondissements) comme polygones
zones_pdf = df_zones.toPandas()
for _, row in zones_pdf.drop_duplicates("zone_id").iterrows():
    folium.CircleMarker(
        location=[row["centroid_lat"], row["centroid_lon"]],
        radius=30,
        color="#1D9E75",
        fill=True,
        fill_opacity=0.15,
        tooltip=f"Zone {row['zone_id']} — {row['zone_name']} ({row['zone_type']})"
    ).add_to(m)
 
    folium.Marker(
        location=[row["centroid_lat"], row["centroid_lon"]],
        icon=folium.DivIcon(
            html=f'<div style="font-size:10px;font-weight:bold;color:#0F6E56">{row["zone_id"]}</div>'
        )
    ).add_to(m)
 
# Ajouter les points de départ des trajets remappés
for _, row in points_pdf.iterrows():
    if row["start_casa"] is not None:
        call_type = str(row["CALL_TYPE"])
        color = colors_call.get(call_type, "gray")
        folium.CircleMarker(
            location=row["start_casa"],
            radius=3,
            color=color,
            fill=True,
            fill_opacity=0.7,
            tooltip=f"Taxi {row['TAXI_ID']} | Type {call_type}"
        ).add_to(m)
 
# Légende
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background: white; padding: 10px; border: 1px solid #ccc; border-radius: 8px;
     font-size: 12px;">
  <b>Type de course</b><br>
  <span style="color:blue">●</span> A — Dispatché<br>
  <span style="color:green">●</span> B — Station<br>
  <span style="color:red">●</span> C — Rue<br>
  <span style="color:#1D9E75">○</span> Zones Casa
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
# Sauvegarder la carte
m.save("/home/jovyan/work/notebooks/casablanca_remapped_trips.html")
print("Carte sauvegardée : casablanca_remapped_trips.html")
print("Ouvre ce fichier dans ton navigateur pour voir les trajets sur Casablanca")
m

Carte sauvegardée : casablanca_remapped_trips.html
Ouvre ce fichier dans ton navigateur pour voir les trajets sur Casablanca


CELLULE 7 — Validation du remapping

In [9]:
# Vérifier que tous les points transformés sont bien dans la bbox Casablanca
 
def validate_point(polyline_str):
    """Vérifie que tous les points sont dans la bbox Casa"""
    if not polyline_str:
        return True
    try:
        pts = json.loads(polyline_str)
        for pt in pts:
            lon, lat = pt[0], pt[1]
            if not (CASA_LON_MIN <= lon <= CASA_LON_MAX and
                    CASA_LAT_MIN <= lat <= CASA_LAT_MAX):
                return False
        return True
    except:
        return False
 
validate_udf = F.udf(validate_point, BooleanType())
 
df_validated = df_remapped.withColumn("valid", validate_udf(F.col("POLYLINE_CASA")))
n_total = df_validated.count()
n_valid = df_validated.filter(F.col("valid") == True).count()
n_invalid = n_total - n_valid
 
print(f"Total trajets testés : {n_total:,}")
print(f"Points dans bbox Casa : {n_valid:,} ({n_valid/n_total*100:.1f}%)")
print(f"Points hors bbox     : {n_invalid:,} ({n_invalid/n_total*100:.1f}%)")
 
if n_invalid / n_total > 0.02:
    print("ATTENTION : plus de 2% de points hors bbox — vérifier la transformation")
else:
    print("OK : remapping validé")

Total trajets testés : 8,366
Points dans bbox Casa : 7,726 (92.3%)
Points hors bbox     : 640 (7.7%)
ATTENTION : plus de 2% de points hors bbox — vérifier la transformation
